# Worked Example: Time-Frequency Reward Contrast

## Goal
Morlet TFR with cross-event trialwise baseline normalization using `feedback_start` task epochs and `baseline_start` baseline epochs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from LFPAnalysis import build_event_locked_pipeline_config, run_pipeline

beh = pd.read_csv(Path('../../data/sample_beh.csv'))
chan = 'racas1-racas2'
freqs = np.arange(4, 30, 4).tolist()
config = build_event_locked_pipeline_config(
    Path('../../data/sample_ieeg_bp.fif'),
    file_format='mne',
    event_name='feedback_start',
    event_times=beh['feedback_start'].tolist(),
    baseline_mode='trialwise',
    baseline_event_times=beh['baseline_start'].tolist(),
    baseline_window=(-0.5, 0.0),
    tmin=-0.5,
    tmax=1.5,
    metadata={'reward': beh['reward'].tolist(), 'rpe': beh['rpe'].tolist()},
    tfr_method='morlet',
    tfr_freqs=freqs,
    tfr_n_cycles=3.0,
)
result = run_pipeline(config)
power = result.tfr['power'].copy().pick([chan])
reward_ix = np.where(power.metadata['reward'].to_numpy() == 1)[0]
loss_ix = np.where(power.metadata['reward'].to_numpy() == 0)[0]
reward_map = np.nanmean(power.data[reward_ix, 0], axis=0)
loss_map = np.nanmean(power.data[loss_ix, 0], axis=0)
diff = reward_map - loss_map
print('TFR shape:', power.data.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
freq_arr = np.asarray(freqs)
for ax, data, title in zip(
    axes,
    [reward_map, loss_map, diff],
    ['reward', 'no reward', 'difference'],
):
    im = ax.imshow(
        data,
        aspect='auto',
        origin='lower',
        extent=[power.times[0], power.times[-1], freq_arr[0], freq_arr[-1]],
        cmap='RdBu_r',
    )
    ax.axvline(0, color='k', ls='--', lw=0.8)
    ax.set(xlabel='Time (s)', title=title)
axes[0].set_ylabel('Frequency (Hz)')
fig.colorbar(im, ax=axes, shrink=0.8, label='Trialwise z-scored power')
fig.tight_layout()
plt.show()

## Saving results

See chapter 15 (`15_saving_and_organizing_results`) for the recommended `results/` layout.

In [ ]:
# Uncomment to save. See chapter 15 for the recommended results/ layout.
# out = Path('../../results/worked-examples')
# out.mkdir(parents=True, exist_ok=True)
# power.save(out / 'feedback_cross_event_trialwise-tfr.h5', overwrite=True)
# Reload with: mne.time_frequency.read_tfrs(out / 'feedback_cross_event_trialwise-tfr.h5')

## Next step

Chapter 10 (`10_first_connectivity_and_surrogates`) covers connectivity.